# Fine-tune SpeechBrain Tacotron 2 on Your Voice — 200 Sentences

This notebook prepares **200 WAV recordings and their transcripts**, fine-tunes the pretrained SpeechBrain Tacotron 2 model, and synthesizes a test WAV.

**Expected data format**
- Audio files: `001.wav` through `200.wav`
- Metadata: pipe-delimited, no header, three columns: `ID|transcript|normalized_transcript`
- Audio: mono WAV at 22050 Hz is recommended to match the LJSpeech pretrained model.

**Kaggle setup:** Attach the Kaggle dataset containing your 200 WAV files and the dataset containing the 200-row metadata file. In Section 2, update `AUDIO_DIR` and `SOURCE_METADATA` if the mounted Kaggle paths differ from the defaults. If you use the generated metadata file, upload `metadata_001-200.csv` to Kaggle and point `SOURCE_METADATA` to its mounted path.

Run the cells from top to bottom in a fresh Kaggle session. Enable a GPU in Notebook Settings. The initial settings are 15 epochs, batch size 4, and learning rate `5e-5`, matching the earlier setup as a starting point. With 200 sentences, the notebook creates a deterministic 180/20 train/validation split.


## 1. Install dependencies and load pretrained models

In [ ]:
%pip install -q speechbrain soundfile tgt Unidecode

import torch
import soundfile as sf
import pandas as pd
from pathlib import Path
from IPython.display import Audio, display
from speechbrain.inference.TTS import Tacotron2
from speechbrain.inference.vocoders import HIFIGAN

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# Download/cache the pretrained model files used for initialization and vocoding.
tacotron2 = Tacotron2.from_hparams(
    source="speechbrain/tts-tacotron2-ljspeech",
    savedir="/kaggle/working/pretrained_tacotron2",
    run_opts={"device": DEVICE},
)
hifi_gan = HIFIGAN.from_hparams(
    source="speechbrain/tts-hifigan-ljspeech",
    savedir="/kaggle/working/pretrained_hifigan",
    run_opts={"device": DEVICE},
)

PRETRAINED_CKPT = Path("/kaggle/working/pretrained_tacotron2/model.ckpt")
assert PRETRAINED_CKPT.is_file(), f"Pretrained checkpoint not found: {PRETRAINED_CKPT}"
print("Pretrained Tacotron 2 checkpoint:", PRETRAINED_CKPT)


## 2. Prepare the LJSpeech-style dataset

In [ ]:
# Change this to the mounted folder containing 001.wav ... 200.wav.
AUDIO_DIR = Path("/kaggle/input/datasets/YOUR_USERNAME/speech-set1")
TRANSCRIPT_DIR = Path("/kaggle/input/datasets/YOUR_USERNAME/transcript")
# Point this to the uploaded 200-row metadata file (e.g. metadata_001-200.csv).
SOURCE_METADATA = TRANSCRIPT_DIR / "metadata_001-200.csv"

WORK_DIR = Path("/kaggle/working/my_voice_tts_200")
DATA_DIR = WORK_DIR / "LJSpeech-1.1"
WAV_DIR = DATA_DIR / "wavs"
OUTPUT_DIR = WORK_DIR / "results"

assert AUDIO_DIR.is_dir(), f"Audio input directory not found: {AUDIO_DIR}"
if not SOURCE_METADATA.is_file():
    # Backward-compatible fallback if the uploaded 200-row file is named metadata.csv.
    fallback_metadata = TRANSCRIPT_DIR / "metadata.csv"
    if fallback_metadata.is_file():
        SOURCE_METADATA = fallback_metadata
    else:
        raise FileNotFoundError(
            f"Transcript metadata not found: {SOURCE_METADATA}. "
            "Update SOURCE_METADATA to the mounted path of your 200-row metadata file."
        )

metadata = pd.read_csv(
    SOURCE_METADATA,
    sep="|",
    header=None,
    names=["filename", "transcript", "normalized_transcript"],
    dtype=str,
).fillna("")

for col in ["filename", "transcript", "normalized_transcript"]:
    metadata[col] = metadata[col].str.strip()

assert len(metadata) == 200, f"Expected 200 metadata rows, found {len(metadata)}"
assert metadata["filename"].is_unique, "Duplicate utterance IDs in metadata"

WAV_DIR.mkdir(parents=True, exist_ok=True)
for row in metadata.itertuples(index=False):
    source_wav = AUDIO_DIR / f"{row.filename}.wav"
    if not source_wav.is_file():
        raise FileNotFoundError(f"Missing audio file: {source_wav}")
    # Copy only if absent, so rerunning this cell is safe.
    destination = WAV_DIR / source_wav.name
    if not destination.exists():
        import shutil
        shutil.copy2(source_wav, destination)

# Save the LJSpeech-style pipe-delimited metadata file.
metadata.to_csv(DATA_DIR / "metadata.csv", sep="|", header=False, index=False)

wav_files = list(WAV_DIR.glob("*.wav"))
total_minutes = sum(sf.info(str(p)).duration for p in wav_files) / 60
print("Dataset directory:", DATA_DIR)
print("WAV files:", len(wav_files))
print("Metadata rows:", len(metadata))
print("Total audio duration (minutes):", round(total_minutes, 2))
print("Sample rate counts:")
print(pd.Series([sf.info(str(p)).samplerate for p in wav_files]).value_counts().sort_index())

assert len(wav_files) == 200, f"Expected 200 WAVs, found {len(wav_files)}"


## 3. Get the SpeechBrain recipe and prepare the fine-tuning script

In [ ]:
import subprocess
import sys

REPO = Path("/kaggle/working/speechbrain_repo")
REPO_URL = "https://github.com/speechbrain/speechbrain.git"

if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO)],
        check=True,
    )

RECIPE_DIR = REPO / "recipes/LJSpeech/TTS/tacotron2"
TRAIN_SCRIPT = RECIPE_DIR / "train.py"
HYPERPARAMS = RECIPE_DIR / "hparams/train.yaml"
CUSTOM_TRAIN_SCRIPT = WORK_DIR / "train_finetune.py"
PREP_HELPER = RECIPE_DIR / "ljspeech_prepare.py"

assert TRAIN_SCRIPT.is_file(), f"Training recipe missing: {TRAIN_SCRIPT}"
assert HYPERPARAMS.is_file(), f"Hyperparameters missing: {HYPERPARAMS}"
assert PREP_HELPER.is_file(), f"Dataset preparation helper missing: {PREP_HELPER}"

# Keep the repository's original train.py untouched; patch a separate copy
# so it initializes the recipe model with the pretrained Tacotron 2 weights.
source = TRAIN_SCRIPT.read_text(encoding="utf-8")
needle = "        hparams = load_hyperpyyaml(fin, overrides)"
insertion = '''        hparams = load_hyperpyyaml(fin, overrides)

    # Initialize Tacotron 2 with the pretrained SpeechBrain weights.
    pretrained_path = "/kaggle/working/pretrained_tacotron2/model.ckpt"
    pretrained_state = torch.load(
        pretrained_path, map_location="cpu", weights_only=True
    )
    result = hparams["model"].load_state_dict(pretrained_state, strict=True)
    print("Pretrained Tacotron 2 weights loaded:", result)
'''

if "Initialize Tacotron 2 with the pretrained SpeechBrain weights." not in source:
    if source.count(needle) != 1:
        raise RuntimeError("Could not safely find the model initialization point in train.py.")
    source = source.replace(needle, insertion, 1)

# The recipe imports its helper by module name; add the recipe directory to sys.path.
path_setup = f"import sys\nsys.path.insert(0, {str(RECIPE_DIR)!r})\n"
if not source.startswith(path_setup):
    source = path_setup + source

CUSTOM_TRAIN_SCRIPT.write_text(source, encoding="utf-8")

print("Custom training script:", CUSTOM_TRAIN_SCRIPT)
print("Recipe config:", HYPERPARAMS)


## 4. Create verified 90/10 train/validation manifests (180/20)

In [ ]:
import csv
import json
import random

SAVE_DIR = OUTPUT_DIR / "save"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

rows = []
with open(DATA_DIR / "metadata.csv", "r", encoding="utf-8", newline="") as f:
    for row in csv.reader(f, delimiter="|"):
        if not row or len(row) < 2:
            continue
        utt_id = row[0].strip()
        transcript = (row[2] if len(row) >= 3 and row[2].strip() else row[1]).strip()
        wav_path = WAV_DIR / f"{utt_id}.wav"

        if not transcript:
            raise ValueError(f"Empty transcript for {utt_id}")
        if not wav_path.is_file():
            raise FileNotFoundError(f"Missing WAV: {wav_path}")

        rows.append({
            "uttid": utt_id,
            "wav": str(wav_path.resolve()),
            "label": transcript,
            "segment": False,
            "duration": sf.info(str(wav_path)).duration,
        })

if len(rows) != 200:
    raise RuntimeError(f"Expected 200 matched audio/transcript records; found {len(rows)}")

random.Random(1234).shuffle(rows)
split_index = int(len(rows) * 0.90)  # 180 train, 20 validation for 200 records
train_records, valid_records = rows[:split_index], rows[split_index:]

def write_manifest(filename, records):
    path = SAVE_DIR / filename
    with open(path, "w", encoding="utf-8") as f:
        json.dump({r["uttid"]: r for r in records}, f, ensure_ascii=False, indent=2)
    return path

train_path = write_manifest("train.json", train_records)
valid_path = write_manifest("valid.json", valid_records)

assert len(json.loads(train_path.read_text(encoding="utf-8"))) == split_index
assert len(json.loads(valid_path.read_text(encoding="utf-8"))) == len(rows) - split_index
print("Train manifest:", train_path, f"| {len(train_records)} records")
print("Validation manifest:", valid_path, f"| {len(valid_records)} records")


## 5. Fine-tune Tacotron 2

In [ ]:
# This is the training run. It uses the pretrained checkpoint injected above.
# skip_prep=True preserves the verified manifests created in the previous cell.

command = [
    sys.executable,
    str(CUSTOM_TRAIN_SCRIPT),
    str(HYPERPARAMS),
    "--device=" + ("cuda:0" if torch.cuda.is_available() else "cpu"),
    "--max_grad_norm=1.0",
    f"--data_folder={DATA_DIR}",
    f"--output_folder={OUTPUT_DIR}",
    "--skip_prep=True",
    "--epochs=15",
    "--learning_rate=0.00005",
    "--batch_size=4",
]

print("Starting fine-tuning...")
print(f"Training examples: {len(train_records)} | Validation examples: {len(valid_records)}")
print("Epochs: 15 | Batch size: 4 | Learning rate: 0.00005")

result = subprocess.run(
    command,
    cwd=str(RECIPE_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(result.stdout[-20000:])
print("Exit code:", result.returncode)
if result.returncode != 0:
    raise RuntimeError("Training failed. Review the training log above.")
print("Training process completed.")


## 6. Load the latest fine-tuned checkpoint and synthesize speech

In [ ]:
# Find the most recent SpeechBrain checkpoint created by training.
checkpoint_dirs = sorted(
    [p for p in SAVE_DIR.glob("CKPT*") if (p / "model.ckpt").is_file()],
    key=lambda p: p.name,
)
if not checkpoint_dirs:
    raise FileNotFoundError(f"No fine-tuned model checkpoint found under {SAVE_DIR}")

FINETUNED_CKPT = checkpoint_dirs[-1] / "model.ckpt"
print("Using fine-tuned checkpoint:", FINETUNED_CKPT)

state = torch.load(FINETUNED_CKPT, map_location="cpu", weights_only=True)
load_result = tacotron2.mods["model"].load_state_dict(state, strict=True)
print("Checkpoint loading:", load_result)
tacotron2.eval()

test_text = (
    "Hello, this is a test of my personalized text to speech system. "
    "I am checking how naturally it reproduces my voice."
)

with torch.inference_mode():
    mel_output, mel_length, alignment = tacotron2.encode_text(test_text)
    waveform = hifi_gan.decode_batch(mel_output)

audio = waveform[0].detach().cpu().squeeze().numpy()
OUTPUT_WAV = WORK_DIR / "fine_tuned_test.wav"
sf.write(str(OUTPUT_WAV), audio, 22050)

print("Generated WAV:", OUTPUT_WAV)
print("Duration:", round(len(audio) / 22050, 2), "seconds")
display(Audio(audio, rate=22050))


## Notes

- To synthesize a different sentence, change `test_text` in the last cell and rerun that cell.
- The final WAV is saved at `/kaggle/working/my_voice_tts_200/fine_tuned_test.wav`.
- The fine-tuned checkpoint is saved under `/kaggle/working/my_voice_tts_200/results/save/`.
- This notebook recreates the successful workflow. It does not include exploratory diagnostics or failed attempts from the original notebook.
- For a fresh run, Kaggle needs network access to install packages, clone SpeechBrain, and download the pretrained checkpoints.

- The dataset preparation cell verifies that there are exactly 200 metadata rows and that each matching WAV exists.
- This notebook starts from the original pretrained LJSpeech Tacotron 2 checkpoint, not the checkpoint fine-tuned on the earlier 100 sentences.
- The 200-sentence dataset is split into 180 training and 20 validation examples. With a small dataset, listen to several held-out examples and avoid assuming that more epochs always improve voice similarity.
